In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input

from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              f1_score, precision_score, recall_score, accuracy_score,
                              average_precision_score)

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ Keras version: {tf.keras.__version__ if hasattr(tf.keras, '__version__') else 'bundled with TF'}")
# Check if GPU is available (it probably isn't, but CPU works fine for this)
gpu_available = len(tf.config.list_physical_devices('GPU')) > 0
print(f"GPU available: {gpu_available}")
print("(CPU training is fine — this dataset is small enough)")

In [ ]:
# We use the SMOTE-balanced training data here. Neural nets really benefit 
# from balanced training sets because they don't have built-in imbalance handling.

X_train = joblib.load('../data/processed/X_train.pkl').values
y_train = joblib.load('../data/processed/y_train.pkl').values
X_test  = joblib.load('../data/processed/X_test.pkl').values
y_test  = joblib.load('../data/processed/y_test.pkl').values

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Train fraud ratio: {y_train.mean()*100:.2f}% (balanced by SMOTE)")
print(f"Test fraud ratio:  {y_test.mean()*100:.2f}% (real-world)")

In [ ]:
# Deep Neural Network with 3 hidden layers.
# - BatchNormalization: stabilizes training
# - Dropout: prevents overfitting (randomly zeros 30% of neurons)
# - ReLU activation: the workhorse of deep learning
# - Sigmoid at the end: outputs a probability between 0 and 1

def build_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        
        Dense(16, activation='relu'),
        
        Dense(1, activation='sigmoid')  # binary output: P(fraud)
    ])
    return model

model = build_model(X_train.shape[1])

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc'),
        keras.metrics.AUC(name='pr_auc', curve='PR')
    ]
)

model.summary()

In [ ]:
# Smart callbacks make training efficient:
# - EarlyStopping: stop training if validation loss doesn't improve
# - ReduceLROnPlateau: lower learning rate when stuck
# - ModelCheckpoint: save the best model along the way

early_stop = callbacks.EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print("🚀 Training Neural Network...")
start = time.time()

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=512,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

elapsed = time.time() - start
print(f"\n✅ Training done in {elapsed:.1f} sec ({elapsed/60:.1f} min)")
print(f"Stopped at epoch {len(history.history['loss'])}")

In [ ]:
# Visualize how training progressed
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', color='#3498db', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#e74c3c', linewidth=2)
axes[0].set_title('Training Loss', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend()

# AUC
axes[1].plot(history.history['auc'], label='Train AUC', color='#3498db', linewidth=2)
axes[1].plot(history.history['val_auc'], label='Val AUC', color='#e74c3c', linewidth=2)
axes[1].set_title('ROC-AUC', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()

# PR-AUC
axes[2].plot(history.history['pr_auc'], label='Train PR-AUC', color='#3498db', linewidth=2)
axes[2].plot(history.history['val_pr_auc'], label='Val PR-AUC', color='#e74c3c', linewidth=2)
axes[2].set_title('PR-AUC', fontweight='bold', fontsize=13)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('PR-AUC')
axes[2].legend()

plt.tight_layout()
plt.savefig('../reports/10_nn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Get predictions
y_proba = model.predict(X_test, batch_size=512, verbose=0).flatten()
y_pred = (y_proba >= 0.5).astype(int)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_proba)
pr_auc    = average_precision_score(y_test, y_proba)

print("=" * 60)
print("🧠 NEURAL NETWORK — TEST SET PERFORMANCE")
print("=" * 60)
print(f"  Accuracy:     {accuracy*100:.3f}%")
print(f"  Precision:    {precision*100:.2f}%")
print(f"  Recall:       {recall*100:.2f}%")
print(f"  F1-Score:     {f1:.4f}")
print(f"  ROC-AUC:      {roc_auc:.4f}")
print(f"  PR-AUC:       {pr_auc:.4f}")
print("\n" + classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'],
            cbar=False, annot_kws={"size": 16, "weight": "bold"})
axes[0].set_title('Neural Net — Confusion Matrix', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

tn, fp, fn, tp = cm.ravel()
axes[0].text(0.5, -0.25, f'TN={tn:,}  FP={fp}  FN={fn}  TP={tp}',
             ha='center', transform=axes[0].transAxes, fontsize=10)

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#e67e22', linewidth=2, label=f'NN (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[1].set_title('ROC Curve', fontweight='bold', fontsize=13)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

prec, rec, _ = precision_recall_curve(y_test, y_proba)
axes[2].plot(rec, prec, color='#f39c12', linewidth=2, label=f'NN (PR-AUC = {pr_auc:.4f})')
axes[2].set_title('Precision-Recall Curve', fontweight='bold', fontsize=13)
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend(loc='lower left')

plt.tight_layout()
plt.savefig('../reports/11_nn_performance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save the Keras model
model.save('../models/neural_network.keras')

nn_results = {
    'model_name': 'Neural Network',
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'training_time_sec': elapsed,
    'confusion_matrix': cm.tolist(),
    'epochs_trained': len(history.history['loss']),
}
joblib.dump(nn_results, '../models/nn_results.pkl')

print("💾 Model saved to: models/neural_network.keras")
print("💾 Metrics saved to: models/nn_results.pkl")

# 🏆 FINAL 3-WAY COMPARISON
rf_results = joblib.load('../models/rf_results.pkl')
xgb_results = joblib.load('../models/xgb_results.pkl')

print("\n" + "=" * 80)
print("🏆 FINAL 3-WAY COMPARISON")
print("=" * 80)
print(f"{'Metric':<15}{'Random Forest':<18}{'XGBoost':<18}{'Neural Net':<18}{'Winner'}")
print("-" * 80)
for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'pr_auc']:
    vals = {'RF': rf_results[metric], 'XGB': xgb_results[metric], 'NN': nn_results[metric]}
    winner_key = max(vals, key=vals.get)
    winner_map = {'RF': '🌲 RF', 'XGB': '🏆 XGBoost', 'NN': '🧠 NN'}
    print(f"{metric:<15}{vals['RF']:<18.4f}{vals['XGB']:<18.4f}{vals['NN']:<18.4f}{winner_map[winner_key]}")

